# HPO: Hyperparameter Optimization

다양한 HPO 방법을 적용하여 Baseline Model의 성능을 개선하고, 각 방법의 성능과 탐색 효율을 비교

[HPO Methods]
1. Grid Search
2. Random Search
3. Bayesian Search
4. Optuna (TPE)

[Data]
- Bank Marketing Dataset
- 오류 방지를 위한 최소한의 전처리만 수행


In [ ]:
#  UCI Machine Learning Data Load Library
!pip install ucimlrepo

In [ ]:
#  Bayesian Search Library
!pip install scikit-optimize

In [ ]:
#  Bayesian Search(TPE) Library
!pip install optuna

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, LeaveOneOut
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import time
import xgboost as xgb


## 1. Data Load

- Bank Marketing Dataset Load

In [ ]:
from ucimlrepo import fetch_ucirepo

def load_data():

    # fetch dataset
    bank_marketing = fetch_ucirepo(id=222)

    # data (as pandas dataframes)
    X = bank_marketing.data.features
    y = bank_marketing.data.targets

    # Concatenate
    df = pd.concat([X, y], axis=1)

    print(f"Shape: {df.shape}")
    print(f"Info: {df.info()}")
    print(f"Samples: {df.head(5)}")
    print("Bank Marketing Dataset load")

    return df, X, y

# 1.데이터 로드
df, X, y = load_data()

## 2. EDA

- 주요 변수의 데이터 타입과 분포 확인


In [ ]:
def eda_data(df):

    # 기본 통계
    print("데이터 타입:")
    print(df.dtypes)

    print("\n결측치 확인:")
    print(df.isnull().sum())

    print("\n타겟 변수 분포:")
    target_counts = df['y'].value_counts()
    print(target_counts)
    print(f"비율: {target_counts / len(df) * 100}")

    print("\n숫자형 변수 기본 통계:")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    print(df[numeric_cols].describe())

    print("\n범주형 변수 고유값 개수:")
    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        print(f"{col}: {df[col].nunique()}개 고유값")
        if df[col].nunique() <= 10:  # 고유값이 적으면 분포 출력
            print(df[col].value_counts().head())
        print()

# 2.데이터 탐색
eda_data(df)

## 3. Pre-processing

- 모델 학습을 위한 최소한의 데이터 전처리
- 범주형 변수 변환 및 Train / Test 데이터 구성



In [ ]:
def preprocess_data(X, y):

    # 데이터프레임 복사
    X_processed = X.copy()

    # 범주형 변수들을 레이블 인코딩
    categorical_columns = X_processed.select_dtypes(include=['object']).columns
    label_encoders = {}

    for col in categorical_columns:
        le = LabelEncoder()
        X_processed[col] = le.fit_transform(X_processed[col].astype(str))
        label_encoders[col] = le
        print(f"{col} 인코딩 완료: {len(le.classes_)}개 클래스")

    # 타겟 변수 인코딩 (yes=1, no=0)
    y_processed = (y['y'] == 'yes').astype(int)

    print(f"타겟 분포: {y_processed.value_counts().to_dict()}")

    return X_processed, y_processed, label_encoders


# 3. 데이터 전처리
X_processed, y_processed, label_encoders = preprocess_data(X, y)

In [ ]:
label_encoders

## 4. Baseline Modeling & Evaluation

- 기본 Hyperparameter를 사용하여 Baseline Model 학습
- HPO 적용 전 모델의 기준 성능 확인

**Check**
- Baseline Model의 성능은 어느 정도인가?
- 이후 HPO 결과와 비교할 기준은 무엇인가?


In [ ]:
def build_and_evaluate_model(X, y):
    """XGBoost 모델 구축 및 평가"""
    print("\n=== XGBoost 모델 구축 및 평가 ===")

    # 훈련/테스트 데이터 분할
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print(f"훈련 데이터: {X_train.shape}")
    print(f"테스트 데이터: {X_test.shape}")

    # 특성 스케일링
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # XGBoost 모델 훈련
    xgb_model = xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        early_stopping_rounds=5,
        eval_metric='logloss'
    )

    print("XGBoost Learning...")
    xgb_model.fit(
        X_train_scaled, y_train,
        eval_set=[(X_test_scaled, y_test)],
        verbose=False
    )

    # 예측
    y_pred = xgb_model.predict(X_test_scaled)
    y_pred_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]

    # 평가
    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_pred_proba)

    print(f"\n정확도: {accuracy:.4f}")
    print(f"AUC 점수: {auc_score:.4f}")

    print("\n분류 리포트:")
    print(classification_report(y_test, y_pred))

    print("\n혼동 행렬:")
    print(confusion_matrix(y_test, y_pred))

    # 특성 중요도
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': xgb_model.feature_importances_
    }).sort_values('importance', ascending=False)

    print("\n상위 10개 중요한 특성:")
    print(feature_importance.head(10))

    return xgb_model, scaler, feature_importance

# 4. 모델 구축 및 평가
model, scaler, feature_importance = build_and_evaluate_model(X_processed, y_processed)


## 5. Cross-Validation

- Cross Validation을 통해 모델의 성능을 반복적으로 검증
- 단일 Train / Test 결과가 아닌 안정적인 성능 수준 확인

**Check**
- Fold별 성능에는 어느 정도 차이가 있는가?
- 평균적인 모델 성능은 어느 정도인가?


In [ ]:
def compare_cv_strategies(X, y):
    """다양한 CV 전략 비교"""
    from sklearn.model_selection import KFold, RepeatedStratifiedKFold

    print("\n=== Cross-Validation 전략 비교 ===")

    # 스케일링
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 모델
    xgb_model = xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss'
    )

    # CV 전략들
    cv_strategies = {
    #    'Leave-One-Out (LOOCV)': LeaveOneOut(),
        'KFold (5-fold)': KFold(n_splits=5, shuffle=True, random_state=42),
        'StratifiedKFold (5-fold)': StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        'StratifiedKFold (20-fold)': StratifiedKFold(n_splits=20, shuffle=True, random_state=42)
    }

    results = {}
    for name, cv_strategy in cv_strategies.items():
        scores = cross_val_score(xgb_model, X_scaled, y, cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
        results[name] = {
            'mean': scores.mean(),
            'std': scores.std(),
            'scores': scores
        }
        print(f"{name}: {scores.mean():.4f} (±{scores.std()*2:.4f})")

    return results


results = compare_cv_strategies(X_processed, y_processed)

## 6-1. Grid Search

- 사전에 정의한 모든 Hyperparameter 조합 탐색
- Cross Validation을 통해 최적 조합 선택

**Check**
- 어떤 Hyperparameter 조합이 선택되었는가?
- Baseline 대비 성능이 얼마나 개선되었는가?
- 탐색에는 어느 정도 시간이 필요한가?


In [ ]:
def grid_search_tuning_cv(X, y):
    """Cross-Validation을 활용한 그리드 서치 하이퍼파라미터 튜닝"""
    from sklearn.model_selection import GridSearchCV
    print("\n=== 하이퍼파라미터 튜닝 (GridSearchCV) ===")
    t0 = time.perf_counter()

    # 파라미터 그리드 정의
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.05, 0.1, 0.2],
        'subsample': [0.8, 0.9, 1.0]
    }

    # XGBoost 모델
    xgb_model = xgb.XGBClassifier(
        objective='binary:logistic',
        random_state=42,
        eval_metric='logloss'
    )

    # GridSearchCV 설정
    grid_search = GridSearchCV(
        estimator=xgb_model,     # 최적화를 적용할 모델
        param_grid=param_grid,   # 탐색할 하이퍼파라미터 공간
        scoring='roc_auc',
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        n_jobs=-1,               # 병렬 실행 스레드 수, -1 = CPU 전체 사용
        verbose=1
    )

    # 스케일링
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 그리드 서치 실행
    print("GridSearchCV 실행 중...")
    grid_search.fit(X_scaled, y)

    dt = time.perf_counter() - t0
    print(f"수행 시간: {dt:.3f}s")
    print(f"최적 파라미터: {grid_search.best_params_}")
    print(f"최적 CV 점수: {grid_search.best_score_:.4f}")


    return grid_search.best_estimator_, scaler


best_estimator_, scaler = grid_search_tuning_cv(X_processed, y_processed)


## 6-2. Random Search

- Hyperparameter 조합 중 일부를 무작위로 탐색
- 제한된 탐색 횟수에서 효율적으로 최적 조합 탐색

**Check**
- 어떤 Hyperparameter 조합이 선택되었는가?
- Grid Search와 성능에는 어떤 차이가 있는가?
- 탐색 시간은 얼마나 달라졌는가?


In [ ]:
def random_search_tuning(X, y):
    """Cross-Validation을 활용한 랜덤 서치 하이퍼파라미터 튜닝"""
    from sklearn.model_selection import RandomizedSearchCV
    from scipy.stats import uniform, randint

    t0 = time.perf_counter()

    print("\n=== 하이퍼파라미터 튜닝 (RandomizedSearchCV) ===")

    # 랜덤 서치를 위한 파라미터 분포 정의
    param_distributions = {
        'n_estimators': [50, 100, 200],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.05, 0.1, 0.2],
        'subsample': [0.8, 0.9, 1.0]
      #  'colsample_bytree': uniform(0.6, 0.4),
    }

    # XGBoost 모델
    xgb_model = xgb.XGBClassifier(
        objective='binary:logistic',
        random_state=42,
        eval_metric='logloss'
    )

    # RandomizedSearchCV 설정
    random_search = RandomizedSearchCV(
        estimator=xgb_model,      # 최적화를 적용할 모델
        param_distributions=param_distributions,  # 탐색할 하이퍼파라미터 분포
        n_iter=50,               # 탐색할 반복 횟수
        scoring='roc_auc',
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        n_jobs=-1,                # 병렬 실행 스레드 수, -1 = CPU 전체 사용
        verbose=1,
        random_state=42
    )

    # 스케일링
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 랜덤 서치 실행
    print("RandomizedSearchCV 실행 중...")
    random_search.fit(X_scaled, y)

    dt = time.perf_counter() - t0
    print(f"수행 시간: {dt:.3f}s")
    print(f"최적 파라미터: {random_search.best_params_}")
    print(f"최적 CV 점수: {random_search.best_score_:.4f}")

    # 상위 5개 결과 출력
    print("\n상위 5개 파라미터 조합:")
    results_df = pd.DataFrame(random_search.cv_results_)
    top_5 = results_df.nlargest(5, 'mean_test_score')[['mean_test_score', 'std_test_score', 'params']]

    for idx, (_, row) in enumerate(top_5.iterrows(), 1):
        print(f"{idx}. 점수: {row['mean_test_score']:.4f} (±{row['std_test_score']*2:.4f})")
        print(f"   파라미터: {row['params']}")

    return random_search.best_estimator_, scaler, random_search

best_estimator_, scaler, random_search = random_search_tuning(X_processed, y_processed)


## 6-3. Bayesian Search

- 이전 탐색 결과를 활용하여 다음 Hyperparameter 탐색 영역 결정
- 성능이 좋을 가능성이 높은 영역을 중심으로 탐색

**Check**
- 어떤 Hyperparameter 조합이 선택되었는가?
- Random Search와 비교하여 성능은 어떠한가?
- 탐색 효율에는 어떤 차이가 있는가?


In [ ]:
def bayesian_search_tuning(X, y):
    """베이지안 최적화를 활용한 하이퍼파라미터 튜닝"""
    t0 = time.perf_counter()

    from skopt import BayesSearchCV
    from skopt.space import Real, Integer, Categorical

    print("\n=== 하이퍼파라미터 튜닝 (Bayesian Optimization) ===")

    # 베이지안 최적화를 위한 검색 공간 정의
    # log-uniform 분포: 작은 값 쪽에 더 많은 샘플을 할당 → 학습률은 보통 작은 값이 성능에 큰 영향을 주기 때문에 로그 분포로 탐색하는 게 효율적
    search_space = {
        'n_estimators': Integer(50, 200),
        'max_depth': Integer(4, 8),
        'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
        'subsample': Real(0.8, 1.0)
    }


    # XGBoost 모델
    xgb_model = xgb.XGBClassifier(
        objective='binary:logistic',
        random_state=42,
        eval_metric='logloss'
    )

    # BayesSearchCV 설정
    bayes_search = BayesSearchCV(
        estimator=xgb_model,          # 최적화를 적용할 모델
        search_spaces=search_space,   # 탐색할 하이퍼파라미터 공간 (딕셔너리 형식)
        n_iter=30,                    # 탐색할 반복 횟수(기본값:50), 클수록 더 좋은 해를 찾을 확률이 높지만 시간이 오래 걸림, 30~100
        scoring='roc_auc',            # 평가지표
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        n_jobs=-1,                    # 병렬 실행 스레드 수, -1 = CPU 전체 사용
        random_state=42,
        verbose=0                     # 출력 로그 레벨(0 = 없음, 1 이상 = 진행 상황 출력)
    )

    # 스케일링
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 베이지안 최적화 실행
    print("Bayesian Optimization 실행 중...")
    bayes_search.fit(X_scaled, y)

    dt = time.perf_counter() - t0
    print(f"수행 시간: {dt:.3f}s")

    print(f"최적 파라미터: {bayes_search.best_params_}")
    print(f"최적 CV 점수: {bayes_search.best_score_:.4f}")

    # 최적화 과정 분석
    print(f"\n최적화 과정 분석:")
    print(f"전체 시도 횟수: {len(bayes_search.cv_results_['mean_test_score'])}")

    # 상위 5개 결과 출력
    print("\n상위 5개 파라미터 조합:")
    results_df = pd.DataFrame(bayes_search.cv_results_)
    top_5 = results_df.nlargest(5, 'mean_test_score')[['mean_test_score', 'std_test_score', 'params']]

    for idx, (_, row) in enumerate(top_5.iterrows(), 1):
        print(f"{idx}. 점수: {row['mean_test_score']:.4f} (±{row['std_test_score']*2:.4f})")
        print(f"   파라미터: {row['params']}")

    # 수렴 곡선 데이터 생성
    scores_over_time = []
    best_score_so_far = -np.inf
    for score in bayes_search.cv_results_['mean_test_score']:
        if score > best_score_so_far:
            best_score_so_far = score
        scores_over_time.append(best_score_so_far)

    print(f"\n수렴 분석:")
    print(f"초기 10회 평균 점수: {np.mean(scores_over_time[:10]):.4f}")
    print(f"마지막 10회 최고 점수: {scores_over_time[-1]:.4f}")
    print(f"개선 정도: {(scores_over_time[-1] - np.mean(scores_over_time[:10])):.4f}")

    return bayes_search.best_estimator_, scaler, bayes_search

best_estimator_, scaler, bayes_search = bayesian_search_tuning(X_processed, y_processed)


## 6-4. Optuna (TPE)

- TPE를 활용하여 효율적으로 Hyperparameter 탐색
- 반복적인 Trial을 통해 좋은 Parameter 영역 탐색

**Check**
- 어떤 Hyperparameter 조합이 선택되었는가?
- 기존 HPO 방법과 성능에는 어떤 차이가 있는가?
- 성능과 탐색 시간을 함께 고려하면 어떤 특징이 있는가?


In [ ]:
import optuna

def optuna_search_tuning(X, y, n_trials=100):
    """Optuna를 활용한 하이퍼파라미터 튜닝"""
    t0 = time.perf_counter()

    print("\n=== 하이퍼파라미터 튜닝 (Optuna) ===")

    # 전역 변수로 데이터를 저장 (objective 함수에서 사용하기 위해)
    global X_train_global, X_val_global, y_train_global, y_val_global, scaler_global

    # 데이터 분할 및 스케일링
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # 전역 변수에 할당
    X_train_global, X_val_global = X_train_scaled, X_val_scaled
    y_train_global, y_val_global = y_train, y_val
    scaler_global = scaler

    def objective(trial):
        """Optuna objective 함수"""

        # 하이퍼파라미터 제안
        params = {
            'objective': 'binary:logistic',
            'eval_metric': 'logloss',
            'random_state': 42,
            'verbosity': 0,

            # 튜닝할 하이퍼파라미터들
            'n_estimators': trial.suggest_int('n_estimators', 50, 200),
            'max_depth': trial.suggest_int('max_depth', 4, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.8, 1.0)
               }

        # XGBoost 모델 생성 및 훈련
        model = xgb.XGBClassifier(**params)

        model.fit(
            X_train_global, y_train_global,
            eval_set=[(X_val_global, y_val_global)],
            verbose=False
        )

        # 검증 성능 계산
        y_pred_proba = model.predict_proba(X_val_global)[:, 1]
        auc_score = roc_auc_score(y_val_global, y_pred_proba)

        return auc_score

    # Optuna 스터디 생성
    print(f"Optuna 최적화 시작 ({n_trials}회 시도)...")

    # 로그 출력 줄이기
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    )

    # 최적화 실행
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    # 결과 출력
    dt = time.perf_counter() - t0
    print(f"수행 시간: {dt:.3f}s")
    print(f"최적 AUC 점수: {study.best_value:.4f}")
    print(f"최적 파라미터: {study.best_params}")
    print(f"시도 횟수: {len(study.trials)}")

    # 최적 모델 재훈련
    best_params = study.best_params.copy()
    best_params.update({
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'random_state': 42
    })

    best_model = xgb.XGBClassifier(**best_params)
    best_model.fit(X_train_global, y_train_global)

    # 최적화 과정 분석
    print(f"\n=== 최적화 과정 분석 ===")

    # 상위 5개 시도 결과
    best_trials = sorted(study.trials, key=lambda t: t.value if t.value else -np.inf, reverse=True)[:5]
    print("상위 5개 시도 결과:")
    for i, trial in enumerate(best_trials, 1):
        if trial.value:
            print(f"{i}. AUC: {trial.value:.4f}, 파라미터: {trial.params}")

    # 수렴 분석
    values = [trial.value for trial in study.trials if trial.value is not None]
    if len(values) >= 10:
        early_avg = np.mean(values[:len(values)//2])
        late_avg = np.mean(values[len(values)//2:])
        print(f"\n수렴 분석:")
        print(f"전반부 평균 AUC: {early_avg:.4f}")
        print(f"후반부 평균 AUC: {late_avg:.4f}")
        print(f"개선 정도: {late_avg - early_avg:.4f}")

    # 파라미터 중요도 분석
    importance = optuna.importance.get_param_importances(study)
    print(f"\n파라미터 중요도 (Top 5):")
    for param, imp in sorted(importance.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {param}: {imp:.4f}")

    return best_model, scaler_global, study

best_model, scaler_global, study = optuna_search_tuning(X_processed, y_processed, n_trials=30)

## Team Activity — Compare HPO Methods

각자 수행한 실습 결과를 조원들과 비교하고 의견을 나눠봅니다.

[Discussion]
- 어떤 HPO 방법에서 가장 높은 성능이 나왔는가?
- Baseline 대비 성능은 얼마나 개선되었는가?
- HPO 방법별 탐색 시간에는 어떤 차이가 있었는가?
- 실제 업무라면 어떤 HPO 방법을 선택하겠는가?

---

## Challenge 1 — Try Another Model

XGBoost 이외의 모델에 HPO를 적용해 봅니다.

- 적용할 모델 자유롭게 선택
- HPO Method 중 2개 방법 선택
- 동일한 데이터를 사용하여 HPO 수행 및 결과 비교

Challenge
- 모델이 달라지면 최적화 효과도 달라지는가?


In [ ]:
#...

## Challenge 2 — Try Another Dataset

새로운 데이터에서도 HPO를 적용해 봅니다

- Adult (Census Income) Dataset 활용
- 적용 모델 자유롭게 선택
- HPO Method 중 2개 방법 선택
- HPO 결과 비교

[Challenge]
- 데이터가 달라져도 동일한 HPO 방법이 효과적인가?
- 결과를 바탕으로 판단할때 어떤 방법을 선택하는 것이 적절한가?


In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from ucimlrepo import fetch_ucirepo
import pandas as pd

# UCI에서 데이터 불러오기
adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets

# Concatenate후 살펴보기
df = pd.concat([X, y], axis=1)

print(f"Data Shape: {df.shape}")
print(f"Categorical Variable: {X.select_dtypes(include=['object']).columns.tolist()}")
print(f"Info: {df.info()}")

print(f"{df.head(5)}")